# 03 Inference


## 1. Data


In [1]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

PROCESSED_PATH = ROOT / "data" / "processed" / "weight_perception_bmipct_cleaned.csv"
TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

analysis = pd.read_csv(PROCESSED_PATH)

required_cols = ["PerceptionOfWeight", "WeightPerception", "BMIPCT"]
missing_cols = [col for col in required_cols if col not in analysis.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Please rerun 02_EDA_Main.ipynb first.")

weight_perception_labels = {
    1: "Very underweight",
    2: "Slightly underweight",
    3: "About the right weight",
    4: "Slightly overweight",
    5: "Very overweight",
}
weight_perception_order = list(weight_perception_labels.values())

analysis["PerceptionOfWeight"] = analysis["PerceptionOfWeight"].astype(int)
analysis["WeightPerception"] = pd.Categorical(
    analysis["WeightPerception"],
    categories=weight_perception_order,
    ordered=True
)

print("Rows valid for ANOVA analysis:", len(analysis))

Rows valid for ANOVA analysis: 12853


## 2. Method Choice


This project uses a **one-way ANOVA**.

One-way ANOVA is appropriate because:

- The explanatory variable, `PerceptionOfWeight`, is categorical with five groups.
- The response variable, `BMIPCT`, is quantitative.
- The goal is to compare mean BMI percentile across more than two groups.


## 3. Hypotheses

Let:

- `mu_1` = population mean BMIPCT for students who perceive themselves as **Very underweight**
- `mu_2` = population mean BMIPCT for students who perceive themselves as **Slightly underweight**
- `mu_3` = population mean BMIPCT for students who perceive themselves as **About the right weight**
- `mu_4` = population mean BMIPCT for students who perceive themselves as **Slightly overweight**
- `mu_5` = population mean BMIPCT for students who perceive themselves as **Very overweight**

Formal ANOVA hypotheses:

- **H0:** `mu_1 = mu_2 = mu_3 = mu_4 = mu_5`
- **Ha:** Not all group means are equal. Equivalently, `mu_i != mu_j` for at least one pair of groups.

Alpha level: **0.05**.


Hypotheses:

- **H0:** The mean BMIPCT is the same across all PerceptionOfWeight groups.
- **Ha:** At least one PerceptionOfWeight group has a different mean BMIPCT.

Alpha level: **0.05**.


## 4. Assumption Check for One-Way ANOVA


In [2]:
group_summary_for_assumptions = (
    analysis.groupby("WeightPerception", observed=True)["BMIPCT"]
    .agg(n="count", mean="mean", sd="std", median="median")
    .reset_index()
)

group_values = [
    group["BMIPCT"].values
    for _, group in analysis.groupby("WeightPerception", observed=True)
]
levene_stat, levene_p = stats.levene(*group_values, center="median")
levene_p_display = "< 0.0001" if levene_p < 0.0001 else f"{levene_p:.4f}"

assumption_check = pd.DataFrame({
    "assumption question": [
        "Are the groups independent?",
        "Is the response variable quantitative?",
        "Are there at least two groups?",
        "Are group sample sizes usable?",
        "Are group variances exactly equal?",
        "Is ANOVA still reasonable for this class project?",
    ],
    "answer": [
        "Reasonable for this project",
        "Yes",
        "Yes",
        "Yes",
        "Not exactly",
        "Yes, with cautious interpretation",
    ],
    "evidence / explanation": [
        "The dataset is treated as student survey responses for this individual project.",
        "BMIPCT is a numeric BMI percentile variable.",
        "PerceptionOfWeight has five groups.",
        f"Group sample sizes range from {group_summary_for_assumptions['n'].min():,} to {group_summary_for_assumptions['n'].max():,}.",
        f"Levene median-centered test p-value = {levene_p_display}; group standard deviations are not identical.",
        "The sample size is large, and the course-approved method for this research question is one-way ANOVA. The conclusion is stated as association, not causation.",
    ],
})

group_summary_for_assumptions.to_csv(TAB_DIR / "03_anova_group_assumption_summary.csv", index=False)
assumption_check.to_csv(TAB_DIR / "03_anova_assumption_check.csv", index=False)

assumption_check_display = (
    assumption_check.style
    .hide(axis="index")
    .set_properties(**{
        "white-space": "normal",
        "text-align": "left",
        "vertical-align": "top",
        "line-height": "1.3",
    })
    .set_properties(
        subset=["assumption question"],
        **{"min-width": "190px", "max-width": "230px"}
    )
    .set_properties(
        subset=["answer"],
        **{"min-width": "150px", "max-width": "190px"}
    )
    .set_properties(
        subset=["evidence / explanation"],
        **{"min-width": "320px", "max-width": "420px"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("white-space", "normal"),
                ("text-align", "left"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("overflow-wrap", "break-word"),
            ],
        },
    ])
)

display(group_summary_for_assumptions)
display(assumption_check_display)

,WeightPerception,n,mean,sd,median
0,Very underweight,207,45.503699,35.019935,41.492722
1,Slightly underweight,1436,36.533217,25.777645,31.072561
2,About the right weight,7351,59.713900,24.276332,62.642286
3,Slightly overweight,3318,84.988965,16.426505,90.885855
4,Very overweight,541,92.298578,14.953480,98.176458


assumption question,answer,evidence / explanation
Are the groups independent?,Reasonable for this project,The dataset is treated as student survey responses for this individual project.
Is the response variable quantitative?,Yes,BMIPCT is a numeric BMI percentile variable.
Are there at least two groups?,Yes,PerceptionOfWeight has five groups.
Are group sample sizes usable?,Yes,"Group sample sizes range from 207 to 7,351."
Are group variances exactly equal?,Not exactly,Levene median-centered test p-value = < 0.0001; group standard deviations are not identical.
Is ANOVA still reasonable for this class project?,"Yes, with cautious interpretation","The sample size is large, and the course-approved method for this research question is one-way ANOVA. The conclusion is stated as association, not causation."


## 5. One-Way ANOVA


In [3]:
alpha = 0.05

def format_p_value(value, threshold=0.0001):
    if pd.isna(value):
        return ""
    value = float(value)
    return f"< {threshold:.4f}" if value < threshold else f"{value:.4f}"

def format_number(value, decimals=2):
    if pd.isna(value):
        return ""
    return f"{float(value):,.{decimals}f}"

def format_df(value):
    if pd.isna(value):
        return ""
    value = float(value)
    return f"{value:,.0f}" if value.is_integer() else f"{value:,.4f}"

model = smf.ols("BMIPCT ~ C(WeightPerception)", data=analysis).fit()
anova_results = sm.stats.anova_lm(model, typ=2).reset_index()
anova_results = anova_results.rename(
    columns={"index": "source", "PR(>F)": "p_value", "F": "f_statistic"}
)

p_value = anova_results.loc[
    anova_results["source"] == "C(WeightPerception)", "p_value"
].iloc[0]

decision = "Reject H0" if p_value < alpha else "Fail to reject H0"
p_value_display = format_p_value(p_value)

anova_results["alpha"] = alpha
anova_results["decision"] = ""
anova_results.loc[
    anova_results["source"] == "C(WeightPerception)", "decision"
] = decision

anova_results_display = anova_results.copy()
anova_results_display["sum_sq"] = anova_results_display["sum_sq"].apply(lambda value: format_number(value, 2))
anova_results_display["df"] = anova_results_display["df"].apply(format_df)
anova_results_display["f_statistic"] = anova_results_display["f_statistic"].apply(lambda value: format_number(value, 4))
anova_results_display["p_value"] = anova_results_display["p_value"].apply(format_p_value)

anova_results_display.to_csv(TAB_DIR / "anova_results.csv", index=False)
anova_results_display.to_csv(TAB_DIR / "03_anova_results.csv", index=False)

display(anova_results_display)

f_statistic = anova_results.loc[
    anova_results["source"] == "C(WeightPerception)", "f_statistic"
].iloc[0]

print(f"F statistic: {f_statistic:.4f}")
print(f"p-value: {p_value_display}")
print(f"alpha: {alpha}")
print(f"Decision: {decision}")

,source,sum_sq,df,f_statistic,p_value,alpha,decision
0,C(WeightPerception),"3,176,100.27",4,"1,556.6455",< 0.0001,0.05,Reject H0
1,Residual,"6,553,601.38","12,848",,,0.05,


F statistic: 1556.6455
p-value: < 0.0001
alpha: 0.05
Decision: Reject H0


#### Interpretation

The ANOVA tests whether the average BMI percentile is the same across all five perceived weight groups. If the p-value is below 0.05, the result provides evidence that at least one group mean is different.


## 6. Tukey HSD Post-Hoc Comparisons


In [4]:
if p_value < alpha:
    tukey_result = pairwise_tukeyhsd(
        endog=analysis["BMIPCT"],
        groups=analysis["WeightPerception"],
        alpha=alpha
    )
    tukey_results = pd.DataFrame(
        data=tukey_result._results_table.data[1:],
        columns=tukey_result._results_table.data[0]
    )
    tukey_results_display = tukey_results.copy()
    tukey_results_display["p-adj"] = tukey_results_display["p-adj"].apply(format_p_value)
    tukey_results_display.to_csv(TAB_DIR / "tukey_hsd_results.csv", index=False)
    tukey_results_display.to_csv(TAB_DIR / "03_tukey_hsd_results.csv", index=False)
    display(tukey_results_display)
else:
    tukey_results = pd.DataFrame()
    tukey_results_display = pd.DataFrame()
    print("The ANOVA result was not statistically significant, so Tukey HSD was not run.")

,group1,group2,meandiff,p-adj,lower,upper,reject
0,About the right weight,Slightly overweight,25.2751,< 0.0001,23.9864,26.5637,True
1,About the right weight,Slightly underweight,-23.1807,< 0.0001,-24.9584,-21.4030,True
2,About the right weight,Very overweight,32.5847,< 0.0001,29.8399,35.3295,True
3,About the right weight,Very underweight,-14.2102,< 0.0001,-18.5527,-9.8677,True
4,Slightly overweight,Slightly underweight,-48.4557,< 0.0001,-50.4020,-46.5095,True
5,Slightly overweight,Very overweight,7.3096,< 0.0001,4.4527,10.1665,True
6,Slightly overweight,Very underweight,-39.4853,< 0.0001,-43.8994,-35.0711,True
7,Slightly underweight,Very overweight,55.7654,< 0.0001,52.6571,58.8736,True
8,Slightly underweight,Very underweight,8.9705,< 0.0001,4.3896,13.5514,True
9,Very overweight,Very underweight,-46.7949,< 0.0001,-51.8306,-41.7592,True


#### Interpretation

Tukey HSD is used only after a statistically significant ANOVA result. It compares pairs of perceived weight groups while controlling for multiple comparisons.


## 7. Save Inference Summary Table


In [5]:
inference_summary = pd.DataFrame({
    "method": ["One-way ANOVA"],
    "group_variable": ["PerceptionOfWeight / WeightPerception"],
    "response_variable": ["BMIPCT"],
    "f_statistic": [round(float(f_statistic), 4)],
    "p_value": [p_value_display],
    "alpha": [alpha],
    "decision": [decision],
    "post_hoc_test": ["Tukey HSD" if p_value < alpha else "Not used"],
})

inference_summary.to_csv(TAB_DIR / "03_inference_summary.csv", index=False)
display(inference_summary)

,method,group_variable,response_variable,f_statistic,p_value,alpha,decision,post_hoc_test
0,One-way ANOVA,PerceptionOfWeight / WeightPerception,BMIPCT,1556.6455,< 0.0001,0.05,Reject H0,Tukey HSD
